In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
! pip install chromadb

In [ ]:
# ============================================================
# TASK 2.1 — RETRIEVAL PIPELINE
# STRATEGIES: BASELINE COSINE, RE-RANKING, HYDE, MULTI-QUERY
# MODEL: BAAI/BGE-SMALL-EN-V1.5 | DB: CHROMADB
# ============================================================

# ── INSTALL DEPENDENCIES ──────────────────────────────────
# !pip install sentence-transformers chromadb flashrank openai tqdm -q

import json
import os
import time
from typing import Any

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
from openai import OpenAI
from tqdm import tqdm

# ── CONFIG ────────────────────────────────────────────────
CHROMA_PATH   = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/chromadb"  # **CHROMADB PATH ON GOOGLE DRIVE**
QUERIES_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/queries.json"   # **PATH TO QUERIES FILE**
QRELS_PATH    = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/qrels.json"     # **PATH TO QRELS FILE**
EMBED_MODEL   = "BAAI/bge-small-en-v1.5"   # **SAME EMBEDDING MODEL USED DURING INDEXING**
OPENAI_KEY    = "YOUR API KEY"       # **REPLACE WITH YOUR OPENAI API KEY**
TOP_K         = 3    # **NUMBER OF FINAL CHUNKS TO RETURN**
FETCH_K       = 20   # **CANDIDATE POOL SIZE FOR RE-RANKING**

# ── LOAD MODELS & CLIENTS ─────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer(EMBED_MODEL)  # **BGE EMBEDDING MODEL FOR QUERY ENCODING**

print("Loading cross-encoder for re-ranking...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")  # **CROSS-ENCODER FOR RE-RANKING**

print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False)
)  # **PERSISTENT CHROMADB CLIENT POINTING TO GOOGLE DRIVE**


In [ ]:
col_recursive = chroma_client.get_collection("recursive_chunks")  # **RECURSIVE CHUNKING COLLECTION**
col_section   = chroma_client.get_collection("section_chunks")    # **SECTION-AWARE CHUNKING COLLECTION**

openai_client = OpenAI(api_key=OPENAI_KEY)  # **OPENAI CLIENT FOR HYDE AND MULTI-QUERY**

In [ ]:
# ══════════════════════════════════════════════════════════
# HELPER: EMBED A QUERY STRING
# ══════════════════════════════════════════════════════════
def embed_query(text: str) -> list[float]:
    """EMBED A SINGLE QUERY STRING USING BGE MODEL**"""
    return embedder.encode(text, normalize_embeddings=True).tolist()


# ══════════════════════════════════════════════════════════
# HELPER: PARSE RAW CHROMA RESULTS INTO CLEAN CHUNK DICTS
# ══════════════════════════════════════════════════════════
def parse_results(results: dict) -> list[dict]:
    """CONVERT CHROMADB RESULT DICT INTO LIST OF CHUNK DICTS**"""
    chunks = []
    ids        = results["ids"][0]        # **CHUNK IDS**
    documents  = results["documents"][0]  # **CHUNK TEXT**
    metadatas  = results["metadatas"][0]  # **CHUNK METADATA**
    distances  = results["distances"][0]  # **COSINE DISTANCES (LOWER = MORE SIMILAR)**

    for i, chunk_id in enumerate(ids):
        chunks.append({
            "chunk_id":  chunk_id,
            "text":      documents[i],
            "arxiv_id":  metadatas[i].get("arxiv_id", ""),
            "strategy":  metadatas[i].get("strategy", ""),
            "chunk_idx": metadatas[i].get("chunk_index", -1),
            "score":     1 - distances[i],  # **CONVERT DISTANCE TO SIMILARITY SCORE**
        })
    return chunks

In [ ]:
pip install rank_bm25

In [ ]:
# ══════════════════════════════════════════════════════════
# STRATEGY 1 — BASELINE COSINE RETRIEVAL
# ══════════════════════════════════════════════════════════
def retrieve_baseline(query: str, collection, k: int = TOP_K) -> list[dict]:
    """RETRIEVE TOP-K CHUNKS VIA COSINE SIMILARITY BASELINE**"""
    query_vec = embed_query(query)  # **EMBED QUERY WITH BGE**
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )  # **CHROMADB NEAREST NEIGHBOUR LOOKUP**
    return parse_results(results)


# ══════════════════════════════════════════════════════════
# STRATEGY 2 — RE-RANKING (CROSS-ENCODER)
# ══════════════════════════════════════════════════════════
def retrieve_reranked(query: str, collection, k: int = TOP_K, fetch_k: int = FETCH_K) -> list[dict]:
    """RETRIEVE FETCH_K CANDIDATES VIA COSINE THEN RE-RANK WITH CROSS-ENCODER**"""
    # STEP 1: FETCH LARGER CANDIDATE POOL **
    query_vec = embed_query(query)
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"]
    )
    candidates = parse_results(results)  # **CANDIDATE CHUNKS BEFORE RE-RANKING**

    # STEP 2: SCORE EACH CANDIDATE WITH CROSS-ENCODER **
    pairs = [(query, c["text"]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs)  # **CROSS-ENCODER RELEVANCE SCORES**

    # STEP 3: ATTACH SCORES AND SORT DESCENDING **
    for i, chunk in enumerate(candidates):
        chunk["ce_score"] = float(ce_scores[i])

    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:k]  # **RETURN TOP-K AFTER RE-RANKING**


# ══════════════════════════════════════════════════════════
# STRATEGY 3 — HYDE (HYPOTHETICAL DOCUMENT EMBEDDING)
# ══════════════════════════════════════════════════════════

import time

# RETRY WRAPPER FOR OPENAI CALLS **
def safe_openai_call(messages, temperature=0.0, max_tokens=200):
    """RETRY OPENAI CALL UP TO 3 TIMES ON FAILURE**"""
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt+1}/3 failed: {e}")
            if attempt < 2:
                time.sleep(3)  # WAIT 3 SECONDS BEFORE RETRY **
            else:
                print("  ❌ All retries failed, falling back to original query")
                return None   # GIVE UP AFTER 3 ATTEMPTS **

def generate_hypothetical_answer(query: str) -> str:
    """GENERATE HYPOTHETICAL ANSWER WITH RETRY HANDLING**"""
    messages = [
        {
            "role": "system",
            "content": (
                "You are an AI research assistant. Given a question about an AI/ML research paper, "
                "write a short, factual, hypothetical passage that would appear in such a paper "
                "and would directly answer the question. Be concise (2-4 sentences)."
            )
        },
        {"role": "user", "content": query}
    ]
    result = safe_openai_call(messages, temperature=0.0, max_tokens=200)
    return result if result else query  # FALL BACK TO ORIGINAL QUERY IF ALL RETRIES FAIL **



def retrieve_hyde(query: str, collection, k: int = TOP_K) -> list[dict]:
    """EMBED A HYPOTHETICAL ANSWER INSTEAD OF THE RAW QUERY FOR RETRIEVAL**"""
    hyp_answer = generate_hypothetical_answer(query)  # **HYPOTHETICAL PASSAGE**
    hyp_vec = embed_query(hyp_answer)                 # **EMBED HYPOTHETICAL ANSWER**

    results = collection.query(
        query_embeddings=[hyp_vec],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )
    chunks = parse_results(results)

    # ATTACH HYPOTHETICAL ANSWER TO CHUNKS FOR DEBUGGING **
    for c in chunks:
        c["hyde_passage"] = hyp_answer
    return chunks


# ══════════════════════════════════════════════════════════
# STRATEGY 4 — MULTI-QUERY RETRIEVAL
# ══════════════════════════════════════════════════════════
import re

def generate_query_reformulations(query: str) -> list[str]:
    """GENERATE QUERY REFORMULATIONS WITH RETRY HANDLING**"""
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert at query reformulation for academic paper retrieval. "
                "Given a question, generate exactly 4 alternative phrasings. "
                "Return ONLY a JSON array of 4 strings, no preamble or markdown. "
                "Example: [\"phrasing 1\", \"phrasing 2\", \"phrasing 3\", \"phrasing 4\"]"
            )
        },
        {"role": "user", "content": query}
    ]
    result = safe_openai_call(messages, temperature=0.3, max_tokens=300)

    if result is None:
        return [query]  # FALL BACK TO ORIGINAL QUERY **

    # STRIP MARKDOWN CODE FENCES IF PRESENT **
    result = re.sub(r'^```json\s*|\s*```$', '', result.strip())

    try:
        reformulations = json.loads(result)
        if isinstance(reformulations, list):
            return [str(r) for r in reformulations]
        return [query]
    except json.JSONDecodeError:
        print(f"  ⚠️ JSON parse failed, using original query")
        return [query]


def retrieve_multiquery(query: str, collection, k: int = TOP_K) -> list[dict]:
    """RETRIEVE WITH MULTIPLE QUERY REFORMULATIONS AND MERGE RESULTS**"""
    reformulations = generate_query_reformulations(query)
    all_queries = [query] + reformulations  # **ORIGINAL + 4 REFORMULATIONS = 5 TOTAL**

    seen_ids = set()
    merged   = []

    for q in all_queries:
        chunks = retrieve_baseline(q, collection, k=k)
        for chunk in chunks:
            if chunk["chunk_id"] not in seen_ids:
                seen_ids.add(chunk["chunk_id"])
                merged.append(chunk)  # **DEDUPLICATE BY CHUNK ID ACROSS ALL QUERIES**

    # SORT BY COSINE SCORE AND RETURN TOP-K **
    merged_sorted = sorted(merged, key=lambda x: x["score"], reverse=True)
    return merged_sorted[:k]


# ══════════════════════════════════════════════════════════
# STRATEGY 5 — HYBRID SEARCH (BM25 + DENSE + RRF)
# ══════════════════════════════════════════════════════════
from rank_bm25 import BM25Okapi
def build_bm25_index(collection) -> tuple:
    """
    BUILD BM25 INDEX FROM ALL CHUNKS IN A CHROMADB COLLECTION**
    RETURNS: (BM25 OBJECT, LIST OF ALL CHUNKS)**
    CALL ONCE PER COLLECTION AND CACHE**
    """
    print(f"Building BM25 index for {collection.name}...")

    all_data = collection.get(include=["documents", "metadatas"])

    all_chunks = []
    for i, doc in enumerate(all_data["documents"]):
        all_chunks.append({
            "chunk_id":  all_data["ids"][i],
            "text":      doc,
            "arxiv_id":  all_data["metadatas"][i].get("arxiv_id", ""),
            "strategy":  all_data["metadatas"][i].get("strategy", ""),
            "chunk_idx": all_data["metadatas"][i].get("chunk_index", -1),
            "score":     0.0,
        })

    # TOKENIZE CORPUS FOR BM25 — WHITESPACE SPLIT**
    tokenized_corpus = [chunk["text"].lower().split() for chunk in all_chunks]
    bm25 = BM25Okapi(tokenized_corpus)  # **BUILD BM25 INDEX**

    print(f"BM25 index built: {len(all_chunks)} chunks")
    return bm25, all_chunks


def reciprocal_rank_fusion(rankings: list, k: int = 60) -> list:
    """
    COMBINE RANKED LISTS USING RECIPROCAL RANK FUSION**
    RRF SCORE = SUM OF 1/(k + rank) ACROSS ALL LISTS**
    K=60 IS THE STANDARD RRF CONSTANT**
    """
    rrf_scores = {}

    for ranked_list in rankings:
        for rank, chunk_id in enumerate(ranked_list, 1):
            if chunk_id not in rrf_scores:
                rrf_scores[chunk_id] = 0.0
            rrf_scores[chunk_id] += 1.0 / (k + rank)  # **RRF FORMULA**

    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)


def retrieve_hybrid(query: str, collection,
                    k: int = TOP_K,
                    bm25_index = None,
                    all_chunks = None,
                    fetch_k: int = FETCH_K) -> list:
    """
    HYBRID RETRIEVAL: BM25 + DENSE VECTOR SEARCH VIA RRF**
    STEP 1: BM25 KEYWORD SEARCH → RANKED LIST A**
    STEP 2: COSINE VECTOR SEARCH → RANKED LIST B**
    STEP 3: RECIPROCAL RANK FUSION → MERGED RANKING**
    STEP 4: RETURN TOP-K FROM MERGED RANKING**
    """
    if bm25_index is None or all_chunks is None:
        raise ValueError("hybrid requires bm25_index and all_chunks")

    # STEP 1: BM25 KEYWORD SEARCH **
    tokenized_query  = query.lower().split()
    bm25_scores      = bm25_index.get_scores(tokenized_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:fetch_k]
    bm25_ranking     = [all_chunks[i]["chunk_id"] for i in bm25_top_indices]

    # STEP 2: DENSE VECTOR SEARCH **
    query_vec     = embed_query(query)
    dense_results = collection.query(
        query_embeddings=[query_vec],
        n_results=fetch_k,
        include=["documents", "metadatas", "distances"]
    )
    dense_chunks  = parse_results(dense_results)
    dense_ranking = [c["chunk_id"] for c in dense_chunks]

    # STEP 3: RECIPROCAL RANK FUSION **
    fused_ranking = reciprocal_rank_fusion([bm25_ranking, dense_ranking])

    # STEP 4: BUILD RESULT LIST FROM FUSED RANKING **
    chunk_lookup = {c["chunk_id"]: c for c in dense_chunks}
    for idx in bm25_top_indices:
        chunk = all_chunks[idx]
        if chunk["chunk_id"] not in chunk_lookup:
            chunk_lookup[chunk["chunk_id"]] = chunk

    final_chunks = []
    for chunk_id, rrf_score in fused_ranking[:k]:
        if chunk_id in chunk_lookup:
            chunk = chunk_lookup[chunk_id].copy()
            chunk["rrf_score"] = round(rrf_score, 6)
            chunk["score"]     = round(rrf_score, 6)  # **OVERRIDE FOR CONSISTENCY**
            final_chunks.append(chunk)

    return final_chunks[:k]


# ══════════════════════════════════════════════════════════
# BUILD BM25 INDEXES — RUN ONCE, REUSE FOR ALL HYBRID EVALS
# ══════════════════════════════════════════════════════════
print("Building BM25 indexes...")
bm25_recursive, chunks_recursive = build_bm25_index(col_recursive)
bm25_section,   chunks_section   = build_bm25_index(col_section)

In [ ]:
# ══════════════════════════════════════════════════════════
# EVALUATION: HIT RATE @ K
# ══════════════════════════════════════════════════════════
def compute_hit_rate(
    queries:    dict,
    qrels:      dict,
    collection,
    strategy:   str = "baseline",
    k:          int = TOP_K,
    limit:      int = None,
    bm25_index        = None,   # **REQUIRED FOR HYBRID STRATEGY ONLY**
    all_chunks        = None,   # **REQUIRED FOR HYBRID STRATEGY ONLY**
) -> dict:
    """
    COMPUTE HIT RATE@K FOR A GIVEN RETRIEVAL STRATEGY AND COLLECTION**
    A HIT = CORRECT ARXIV_ID APPEARS IN THE TOP-K RETRIEVED CHUNKS**
    SUPPORTS: BASELINE, RERANKED, HYDE, MULTIQUERY, HYBRID**
    """
    hits     = 0
    total    = 0
    failures = []   # **TRACK FAILED QUERIES FOR ERROR ANALYSIS**
    errors   = []   # **TRACK ERRORED QUERIES FOR DEBUGGING**

    query_items = list(queries.items())
    if limit:
        query_items = query_items[:limit]  # **OPTIONAL LIMIT FOR QUICK TESTING**

    for qid, q_data in tqdm(query_items, desc=f"Evaluating [{strategy}]"):
        if qid not in qrels:
            continue  # **SKIP QUERIES WITHOUT GROUND TRUTH**

        query_text   = q_data["query"]
        ground_truth = qrels[qid]["doc_id"]  # **CORRECT ARXIV ID FROM QRELS**

        try:
            # DISPATCH TO CORRECT STRATEGY **
            if strategy == "baseline":
                chunks = retrieve_baseline(query_text, collection, k=k)
            elif strategy == "reranked":
                chunks = retrieve_reranked(query_text, collection, k=k)
            elif strategy == "hyde":
                chunks = retrieve_hyde(query_text, collection, k=k)
            elif strategy == "multiquery":
                chunks = retrieve_multiquery(query_text, collection, k=k)
            elif strategy == "hybrid":
                # HYBRID REQUIRES BM25 INDEX AND CHUNK LIST **
                if bm25_index is None or all_chunks is None:
                    raise ValueError(
                        "hybrid requires bm25_index and all_chunks — "
                        "call build_bm25_index(collection) first"
                    )
                chunks = retrieve_hybrid(
                    query_text, collection,
                    k          = k,
                    bm25_index = bm25_index,
                    all_chunks = all_chunks,
                    fetch_k    = FETCH_K,
                )
            else:
                raise ValueError(f"Unknown strategy: {strategy}")

            retrieved_ids = [c["arxiv_id"] for c in chunks]  # **ARXIV IDS IN TOP-K**
            hit = ground_truth in retrieved_ids               # **CHECK IF CORRECT PAPER RETRIEVED**

            if hit:
                hits += 1
            else:
                failures.append({
                    "query_id":     qid,
                    "query":        query_text,
                    "ground_truth": ground_truth,
                    "retrieved":    retrieved_ids,
                })  # **LOG FAILURES FOR ERROR ANALYSIS**

        except Exception as e:
            # LOG ERROR BUT CONTINUE TO NEXT QUERY **
            errors.append({"query_id": qid, "query": query_text, "error": str(e)})
            print(f"\n  ⚠️ Skipped {qid[:8]}... Error: {e}")
            continue

        total += 1

    hit_rate = hits / total if total > 0 else 0.0
    return {
        "strategy":      strategy,
        "collection":    collection.name,
        "hit_rate_at_k": round(hit_rate, 4),
        "hits":          hits,
        "total":         total,
        "k":             k,
        "failures":      failures,
        "errors":        errors,
    }

In [ ]:
# ══════════════════════════════════════════════════════════
# MAIN — RUN ALL EVALUATIONS
# ══════════════════════════════════════════════════════════

# LOAD BENCHMARK FILES **
print("Loading queries and qrels...")
with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)
with open(QRELS_PATH, "r") as f:
    qrels = json.load(f)
print(f"Total queries: {len(queries)} | Total qrels: {len(qrels)}\n")

# BUILD BM25 INDEXES ONCE — REUSED FOR BOTH HYBRID CONFIGS **
print("Building BM25 indexes for hybrid strategy...")
bm25_recursive, chunks_recursive = build_bm25_index(col_recursive)
bm25_section,   chunks_section   = build_bm25_index(col_section)

results_all = []

LIMIT = None  # **SET TO None FOR FULL 496**

# ALL 10 CONFIGS — 5 STRATEGIES × 2 COLLECTIONS **
configs = [
    # STRATEGY       COLLECTION       BM25_INDEX       ALL_CHUNKS
    ("baseline",   col_recursive, None,           None),
    ("baseline",   col_section,   None,           None),
    ("reranked",   col_recursive, None,           None),
    ("reranked",   col_section,   None,           None),
    ("hyde",       col_recursive, None,           None),
    ("hyde",       col_section,   None,           None),
    ("multiquery", col_recursive, None,           None),
    ("multiquery", col_section,   None,           None),
    ("hybrid",     col_recursive, bm25_recursive, chunks_recursive),  # **BONUS**
    ("hybrid",     col_section,   bm25_section,   chunks_section),    # **BONUS**
]

for strategy, collection, bm25_idx, all_chks in configs:
    print(f"\n{'='*50}")
    print(f"Strategy: {strategy.upper()} | Collection: {collection.name}")
    print(f"{'='*50}")
    result = compute_hit_rate(
        queries, qrels, collection,
        strategy   = strategy,
        k          = TOP_K,
        limit      = LIMIT,
        bm25_index = bm25_idx,   # **NONE FOR NON-HYBRID, INDEX FOR HYBRID**
        all_chunks = all_chks,   # **NONE FOR NON-HYBRID, CHUNKS FOR HYBRID**
    )
    results_all.append(result)
    print(f"Hit Rate@{TOP_K}: {result['hit_rate_at_k']:.4f} "
          f"({result['hits']}/{result['total']})")

# ── PRINT SUMMARY TABLE ───────────────────────────────────
print("\n" + "="*65)
print(f"{'STRATEGY':<14} {'COLLECTION':<22} {'HIT RATE@3':<12} {'HITS':<8} {'TOTAL'}")
print("-"*65)
for r in results_all:
    print(f"{r['strategy']:<14} {r['collection']:<22} "
          f"{r['hit_rate_at_k']:<12.4f} {r['hits']:<8} {r['total']}")


In [ ]:
# ── SAVE RESULTS ──────────────────────────────────────────
output = {
    "task":        "2.1_retrieval_all_strategies",
    "embed_model": EMBED_MODEL,
    "top_k":       TOP_K,
    "fetch_k":     FETCH_K,
    "limit":       LIMIT,
    "results": [
        {k: v for k, v in r.items() if k != "failures"}
        for r in results_all
    ],
    "failure_samples": {
        f"{r['strategy']}_{r['collection']}": r["failures"][:20]
        for r in results_all
    }
}

out_path = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_1_all_results.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\n Results saved to {out_path}")

In [ ]:
import json

file_path = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_1_all_results.json"

with open(file_path, "r") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

In [ ]:
# ============================================================
# TASK 2.2 — GENERATION WITH GROUNDED PROMPTING
# RETRIEVAL: RERANKED | COLLECTION: SECTION_CHUNKS
# LLM: GPT-4O-MINI | CITATIONS: INLINE [SOURCE N]
# ============================================================

import json
import time
import re
from openai import OpenAI
from collections import Counter

# ── CONFIG ────────────────────────────────────────────────
QUERIES_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/queries.json"
ANSWERS_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/answers.json"
QRELS_PATH    = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/qrels.json"
OUTPUT_PATH   = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_2_results.json"
OPENAI_KEY    = "YOUR API KEY"   # **REPLACE THIS**
GEN_MODEL     = "gpt-4o-mini"           # **LLM FOR GENERATION**
TOP_K         = 3                       # **CHUNKS TO RETRIEVE**
TEST_LIMIT    = None                      # **SET TO None FOR FULL 496**

openai_client = OpenAI(api_key=OPENAI_KEY)  # **OPENAI CLIENT**


In [ ]:
# ══════════════════════════════════════════════════════════
# SYSTEM PROMPT
# ══════════════════════════════════════════════════════════
SYSTEM_PROMPT = """You are a precise academic research assistant. Your job is to answer questions about AI/ML research papers.

STRICT RULES:
1. Answer ONLY using information from the provided context chunks below.
2. You MUST cite sources inline using [Source N] notation after every claim.
3. If the context does not contain enough information to answer, respond with exactly: "I don't have enough information in the provided context to answer this question."
4. Do NOT use any prior knowledge or make assumptions beyond what is stated in the context.
5. Keep answers concise and factual.
6. If multiple sources support a claim, cite all of them e.g. [Source 1][Source 2].

FORMAT:
- Write in clear prose
- Use [Source N] immediately after each claim
- End with a "Sources Used:" section listing which sources were cited"""

In [ ]:
# ══════════════════════════════════════════════════════════
# HELPER: FORMAT CONTEXT CHUNKS
# ══════════════════════════════════════════════════════════
def format_context(chunks: list[dict]) -> str:
    """FORMAT RETRIEVED CHUNKS INTO NUMBERED SOURCE BLOCKS**"""
    context_blocks = []
    for i, chunk in enumerate(chunks, 1):
        first_line = chunk["text"].split("\n")[0].strip()
        title = first_line[:80] if first_line.startswith("#") else chunk["arxiv_id"]
        title = title.lstrip("#").strip()
        block = (
            f"[Source {i}: {chunk['arxiv_id']} — {title}]\n"
            f"{chunk['text'][:1000]}"  # **TRUNCATE LONG CHUNKS TO 1000 CHARS**
        )
        context_blocks.append(block)
    return "\n\n---\n\n".join(context_blocks)


# ══════════════════════════════════════════════════════════
# CORE FUNCTION: answer(query) → (str, list[chunk])
# ══════════════════════════════════════════════════════════
def answer(query: str, collection, strategy: str = "reranked") -> tuple[str, list[dict]]:
    """RETRIEVE CHUNKS THEN GENERATE GROUNDED ANSWER WITH CITATIONS**"""

    # STEP 1: RETRIEVE CHUNKS USING CHOSEN STRATEGY **
    if strategy == "reranked":
        chunks = retrieve_reranked(query, collection, k=TOP_K)
    elif strategy == "baseline":
        chunks = retrieve_baseline(query, collection, k=TOP_K)
    elif strategy == "hyde":
        chunks = retrieve_hyde(query, collection, k=TOP_K)
    elif strategy == "multiquery":
        chunks = retrieve_multiquery(query, collection, k=TOP_K)
    elif strategy == "hybrid":
        # Pass the global bm25_section and chunks_section for hybrid strategy
        chunks = retrieve_hybrid(query, collection, k=TOP_K, bm25_index=bm25_section, all_chunks=chunks_section)
    else:
        chunks = retrieve_baseline(query, collection, k=TOP_K)

    # STEP 2: FORMAT CONTEXT BEFORE QUESTION **
    context = format_context(chunks)

    # STEP 3: BUILD USER MESSAGE — CONTEXT FIRST THEN QUESTION **
    user_message = f"""CONTEXT:
{context}

---

QUESTION: {query}

Remember: Answer only from the context above. Cite every claim with [Source N]."""

    # STEP 4: CALL GPT WITH RETRY **
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=GEN_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message}
                ],
                temperature=0.0,    # **TEMPERATURE 0 FOR DETERMINISTIC FACTUAL ANSWERS**
                max_tokens=500
            )
            answer_text = response.choices[0].message.content.strip()
            return answer_text, chunks

        except Exception as e:
            print(f"  ⚠️ Attempt {attempt+1}/3 failed: {e}")
            if attempt < 2:
                time.sleep(3)
            else:
                return "Generation failed due to API error.", chunks

    return "Generation failed.", chunks


# ══════════════════════════════════════════════════════════
# BATCH GENERATION
# ══════════════════════════════════════════════════════════
def run_generation(queries, answers_gt, collection,
                   strategy="reranked", limit=None) -> list[dict]:
    """RUN GENERATION OVER ALL QUERIES AND COLLECT RESULTS FOR RAGAS**"""
    results      = []
    query_items  = list(queries.items())
    if limit:
        query_items = query_items[:limit]

    print(f"Running generation: {len(query_items)} queries | "
          f"strategy={strategy} | collection={collection.name}\n")

    for i, (qid, q_data) in enumerate(query_items):
        query_text   = q_data["query"]
        modality     = q_data.get("source", "text")    # **TEXT, TEXT-TABLE, TEXT-IMAGE**
        gt_answer    = answers_gt.get(qid, "")
        gt_answer    = gt_answer if isinstance(gt_answer, str) else str(gt_answer)

        # GENERATE ANSWER **
        answer_text, chunks = answer(query_text, collection, strategy=strategy)

        results.append({
            "query_id":         qid,
            "query":            query_text,
            "answer":           answer_text,
            "ground_truth":     gt_answer,
            "modality":         modality,
            "contexts":         [c["text"] for c in chunks],   # **FOR RAGAS**
            "retrieved_chunks": [
                {
                    "chunk_id": c["chunk_id"],
                    "arxiv_id": c["arxiv_id"],
                    "text":     c["text"],
                    "score":    c["score"],
                }
                for c in chunks
            ],
        })

        # PRINT PROGRESS EVERY 10 QUERIES **
        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} queries done")

    return results


# ══════════════════════════════════════════════════════════
# DISPLAY HELPER
# ══════════════════════════════════════════════════════════
def display_sample(result: dict):
    """PRETTY PRINT ONE RESULT FOR MANUAL INSPECTION**"""
    print("=" * 60)
    print(f"QUERY:    {result['query']}")
    print(f"MODALITY: {result['modality']}")
    print("-" * 60)
    print(f"ANSWER:\n{result['answer']}")
    print("-" * 60)
    print(f"GROUND TRUTH:\n{result['ground_truth']}")
    print("-" * 60)
    print("RETRIEVED SOURCES:")
    for i, c in enumerate(result["retrieved_chunks"], 1):
        print(f"  [Source {i}] {c['arxiv_id']} | score: {c['score']:.4f}")
        print(f"             {c['text'][:120]}...")
    print("=" * 60)

In [ ]:
# ══════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════
import numpy as np

# LOAD FILES **
print("Loading benchmark files...")
with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)
with open(ANSWERS_PATH, "r") as f:
    answers_gt = json.load(f)

print(f"Queries: {len(queries)} | Answers: {len(answers_gt)}\n")

# ── SANITY CHECK: 25 QUERIES BEFORE FULL RUN ──────────────
print("SANITY CHECK — 25 sample queries")
print("=" * 60)
for qid, q_data in list(queries.items())[:25]:
    ans, chunks = answer(q_data["query"], col_section, strategy="hybrid")
    print(f"\nQ: {q_data['query']}")
    print(f"A: {ans[:400]}")
    print(f"Sources: {[c['arxiv_id'] for c in chunks]}\n")

# ── FULL GENERATION RUN ───────────────────────────────────
print("\nStarting generation run...")
results = run_generation(
    queries    = queries,
    answers_gt = answers_gt,
    collection = col_section,       # **BEST COLLECTION FROM TASK 2.1**
    strategy   = "hybrid",        # **BEST STRATEGY FROM TASK 2.1**
    limit      = TEST_LIMIT
)

# ── DISPLAY 25 SAMPLE RESULTS ─────────────────────────────
print("\n\nSAMPLE RESULTS:")
for r in results[:25]:
    display_sample(r)

# ── MODALITY BREAKDOWN ────────────────────────────────────
print("\nMODALITY BREAKDOWN:")
modality_counts = Counter(r["modality"] for r in results)
for modality, count in sorted(modality_counts.items()):
    print(f"  {modality:<25} {count} queries")

In [ ]:
# ── SAVE ──────────────────────────────────────────────────
output = {
    "task":       "2.2_generation",
    "model":      GEN_MODEL,
    "strategy":   "hybrid",
    "collection": "section_chunks",
    "top_k":      TOP_K,
    "total":      len(results),
    "results":    results,
}
with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"\n Saved {len(results)} results to {OUTPUT_PATH}")

In [ ]:
import json

file_path = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_2_results.json"

with open(file_path, "r") as f:
    data = json.load(f)

print(json.dumps(data, indent=2))

In [ ]:
print("Connecting to ChromaDB...")
chroma_client = chromadb.PersistentClient(
    path=CHROMA_PATH,
    settings=Settings(anonymized_telemetry=False)
)  # **PERSISTENT CHROMADB CLIENT POINTING TO GOOGLE DRIVE**

col_recursive = chroma_client.get_collection("recursive_chunks")  # **RECURSIVE CHUNKING COLLECTION**
col_section   = chroma_client.get_collection("section_chunks")    # **SECTION-AWARE CHUNKING COLLECTION**

In [ ]:
# ============================================================
# TASK 2.3 — ALL PROMPT ENGINEERING EXPERIMENTS
# EXPERIMENT 1: MINIMAL vs ENGINEERED PROMPT
# EXPERIMENT 2: CONTEXT ORDERING (LOST IN THE MIDDLE)
# EXPERIMENT 3: K=3 vs K=5 vs K=10 CHUNKS
# EXPERIMENT 4: CHAIN-OF-THOUGHT vs DIRECT ANSWERING
# METRICS: CUSTOM GPT-4O-MINI JUDGE (NO RAGAS/SPACY/LANGCHAIN)
# ============================================================

import os
import json
import time
import numpy as np
from openai import OpenAI
from collections import Counter

# ── CONFIG ────────────────────────────────────────────────
QUERIES_PATH = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/queries.json"
ANSWERS_PATH = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/answers.json"
OUTPUT_PATH  = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_3_all_experiments.json"
OPENAI_KEY   = "YOUR API KEY"   # **REPLACE WITH YOUR OPENAI API KEY**
GEN_MODEL    = "gpt-4o-mini"           # **LLM FOR ALL EXPERIMENTS**
TEST_LIMIT   = 50                       # **SET TO 50 AFTER SANITY CHECK**

os.environ["OPENAI_API_KEY"] = OPENAI_KEY
openai_client = OpenAI(api_key=OPENAI_KEY)  # **OPENAI CLIENT**



In [ ]:

# ══════════════════════════════════════════════════════════
# CUSTOM METRICS — GPT-4O-MINI AS JUDGE
# NO RAGAS, NO SPACY, NO LANGCHAIN NEEDED
# ══════════════════════════════════════════════════════════
def score_faithfulness(answer: str, contexts: list) -> float:
    """
    SCORE WHETHER ANSWER CLAIMS ARE SUPPORTED BY CONTEXT**
    USES GPT-4O-MINI AS JUDGE — RETURNS FLOAT 0.0 TO 1.0**
    1.0 = FULLY GROUNDED | 0.0 = HALLUCINATED**
    """
    if not answer or answer == "Generation failed.":
        return 0.0

    context_str = "\n\n".join(contexts[:3])  # **USE TOP 3 CONTEXTS**

    prompt = f"""You are evaluating whether an AI answer is faithful to its source context.

CONTEXT:
{context_str[:2000]}

ANSWER:
{answer[:1000]}

Rate how faithful the answer is to the context on a scale of 0.0 to 1.0:
- 1.0 = every claim in the answer is directly supported by the context
- 0.5 = some claims supported, some are not
- 0.0 = answer contains claims not found in context (hallucination)

Respond with ONLY a single decimal number between 0.0 and 1.0. Nothing else."""

    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=5,
            )
            score = float(response.choices[0].message.content.strip())
            return max(0.0, min(1.0, score))  # **CLAMP TO VALID RANGE**
        except Exception as e:
            if attempt < 2:
                time.sleep(2)
            else:
                return 0.0
    return 0.0


def score_answer_relevancy(question: str, answer: str) -> float:
    """
    SCORE WHETHER ANSWER ADDRESSES THE QUESTION ASKED**
    USES GPT-4O-MINI AS JUDGE — RETURNS FLOAT 0.0 TO 1.0**
    1.0 = FULLY RELEVANT | 0.0 = OFF-TOPIC**
    """
    if not answer or answer == "Generation failed.":
        return 0.0

    prompt = f"""You are evaluating whether an AI answer is relevant to the question asked.

QUESTION: {question}

ANSWER: {answer[:1000]}

Rate how relevant the answer is to the question on a scale of 0.0 to 1.0:
- 1.0 = answer directly and completely addresses the question
- 0.5 = answer partially addresses the question
- 0.0 = answer is off-topic or does not address the question

Respond with ONLY a single decimal number between 0.0 and 1.0. Nothing else."""

    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=GEN_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=5,
            )
            score = float(response.choices[0].message.content.strip())
            return max(0.0, min(1.0, score))  # **CLAMP TO VALID RANGE**
        except Exception as e:
            if attempt < 2:
                time.sleep(2)
            else:
                return 0.0
    return 0.0


def run_ragas(results: list, label: str) -> dict:
    """
    EVALUATE RESULTS USING CUSTOM GPT-BASED METRICS**
    FAITHFULNESS: IS ANSWER GROUNDED IN CONTEXT?**
    ANSWER RELEVANCY: DOES ANSWER ADDRESS THE QUESTION?**
    """
    print(f"  📊 Evaluating: {label}...")

    # FILTER FAILED GENERATIONS **
    valid = [r for r in results if r["answer"] != "Generation failed."]
    if len(valid) < len(results):
        print(f"  ⚠️ Skipped {len(results)-len(valid)} failed generations")

    if not valid:
        return {"label": label, "faithfulness": 0.0,
                "answer_relevancy": 0.0, "n": 0}

    faith_scores = []
    relev_scores = []

    for i, r in enumerate(valid):
        # SCORE EACH ANSWER FOR FAITHFULNESS AND RELEVANCY **
        f = score_faithfulness(r["answer"], r["contexts"])
        a = score_answer_relevancy(r["question"], r["answer"])
        faith_scores.append(f)
        relev_scores.append(a)

        if (i + 1) % 5 == 0:
            print(f"    scored {i+1}/{len(valid)}")

    result = {
        "label":            label,
        "faithfulness":     round(float(np.mean(faith_scores)), 4),
        "answer_relevancy": round(float(np.mean(relev_scores)), 4),
        "n":                len(valid),
        "metric_note":      "GPT-4o-mini judge metrics — faithfulness and answer relevancy",
    }
    print(f"     Faithfulness:     {result['faithfulness']:.4f}")
    print(f"     Answer Relevancy: {result['answer_relevancy']:.4f}")
    return result


# ══════════════════════════════════════════════════════════
# SHARED HELPER: CALL GPT WITH RETRY
# ══════════════════════════════════════════════════════════
def call_gpt(system_prompt: str, user_message: str,
             temperature: float = 0.0, max_tokens: int = 600) -> str:
    """CALL GPT-4O-MINI WITH RETRY ON FAILURE**"""
    for attempt in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=GEN_MODEL,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_message},
                ],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            print(f"  ⚠️ Attempt {attempt+1}/3 failed: {e}")
            if attempt < 2:
                time.sleep(3)
            else:
                return "Generation failed."
    return "Generation failed."


# ══════════════════════════════════════════════════════════
# SHARED HELPER: FORMAT CHUNKS
# ══════════════════════════════════════════════════════════
def format_chunks(chunks: list, order: str = "default") -> tuple:
    """
    FORMAT CHUNKS INTO SOURCE BLOCKS**
    ORDER: 'default' = HIGH SCORE FIRST, 'reversed' = LOW SCORE FIRST**
    """
    ordered = list(reversed(chunks)) if order == "reversed" else chunks

    blocks = []
    for i, chunk in enumerate(ordered, 1):
        first_line = chunk["text"].split("\n")[0].strip()
        title = first_line[:80] if first_line.startswith("#") else chunk["arxiv_id"]
        title = title.lstrip("#").strip()
        blocks.append(
            f"[Source {i}: {chunk['arxiv_id']} — {title}]\n"
            f"{chunk['text'][:1000]}"
        )

    context_str  = "\n\n---\n\n".join(blocks)
    context_list = [c["text"] for c in ordered]  # **FLAT LIST FOR SCORING**
    return context_str, context_list


# ══════════════════════════════════════════════════════════
# SHARED HELPER: LOAD QUERY ITEMS
# ══════════════════════════════════════════════════════════
def load_query_items(queries: dict, answers_gt: dict,
                     limit: int = None) -> list:
    """LOAD QUERIES AND PAIR WITH GROUND TRUTH ANSWERS**"""
    items = list(queries.items())
    if limit:
        items = items[:limit]
    return [
        (
            qid,
            q_data["query"],
            q_data.get("source", "text"),
            answers_gt.get(qid, "") if isinstance(answers_gt.get(qid, ""), str)
            else str(answers_gt.get(qid, ""))
        )
        for qid, q_data in items
    ]



In [ ]:

# ══════════════════════════════════════════════════════════
# EXPERIMENT 1 — MINIMAL vs ENGINEERED PROMPT
# ══════════════════════════════════════════════════════════
# HYPOTHESIS: ENGINEERED PROMPT WITH EXPLICIT CITATION RULES
# AND FALLBACK PRODUCES MORE FAITHFUL AND RELEVANT ANSWERS
# THAN A BARE MINIMUM PROMPT**
# INDEPENDENT VARIABLE: SYSTEM PROMPT COMPLEXITY**
# CONTROLLED: SAME QUERIES, CHUNKS, MODEL, TEMPERATURE=0**

MINIMAL_SYSTEM = """You are a helpful assistant. Answer the question using the provided context."""

ENGINEERED_SYSTEM = """You are a precise academic research assistant answering questions about AI/ML research papers.

STRICT RULES:
1. Answer ONLY using information from the provided context chunks.
2. Cite sources inline using [Source N] notation after every claim.
3. If the context lacks sufficient information respond exactly: "I don't have enough information in the provided context to answer this question."
4. Do NOT use prior knowledge beyond what is in the context.
5. Keep answers concise and factual.

FORMAT:
- Write in clear prose
- Use [Source N] after each claim
- End with a "Sources Used:" section"""

def run_experiment_1(query_items: list, collection) -> dict:
    """EXPERIMENT 1: MINIMAL VS ENGINEERED PROMPT**"""
    print("\n" + "="*60)
    print("EXPERIMENT 1 — MINIMAL vs ENGINEERED PROMPT")
    print("="*60)

    minimal_results    = []
    engineered_results = []

    for i, (qid, query, modality, gt) in enumerate(query_items):
        # RETRIEVE ONCE — SAME CHUNKS FOR BOTH CONDITIONS**
        chunks = retrieve_reranked(query, collection, k=3)
        context_str, context_list = format_chunks(chunks, order="default")

        # CONDITION A: MINIMAL PROMPT**
        ans_minimal = call_gpt(
            MINIMAL_SYSTEM,
            f"Context:\n{context_str}\n\nQuestion: {query}"
        )

        # CONDITION B: ENGINEERED PROMPT**
        ans_eng = call_gpt(
            ENGINEERED_SYSTEM,
            f"CONTEXT:\n{context_str}\n\n---\n\nQUESTION: {query}\n\nAnswer only from context. Cite with [Source N]."
        )

        minimal_results.append({
            "question": query, "answer": ans_minimal,
            "contexts": context_list, "ground_truth": gt,
            "modality": modality, "condition": "minimal"
        })
        engineered_results.append({
            "question": query, "answer": ans_eng,
            "contexts": context_list, "ground_truth": gt,
            "modality": modality, "condition": "engineered"
        })

        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} done")

    scores_minimal = run_ragas(minimal_results,    "exp1_minimal")
    scores_eng     = run_ragas(engineered_results, "exp1_engineered")

    return {
        "experiment":           "1_minimal_vs_engineered",
        "hypothesis":           "Engineered prompt produces more faithful and relevant answers",
        "independent_variable": "system prompt complexity",
        "scores":               [scores_minimal, scores_eng],
        "results":              {"minimal":    minimal_results,
                                 "engineered": engineered_results},
    }


# ══════════════════════════════════════════════════════════
# EXPERIMENT 2 — CONTEXT ORDERING (LOST IN THE MIDDLE)
# ══════════════════════════════════════════════════════════
# HYPOTHESIS: MOST RELEVANT CHUNKS FIRST PRODUCES MORE
# FAITHFUL ANSWERS — LLMS ATTEND MORE TO START OF CONTEXT**
# INDEPENDENT VARIABLE: CHUNK ORDER IN CONTEXT**
# CONTROLLED: SAME QUERIES, CHUNKS, PROMPT, MODEL, TEMP=0**

ORDERING_SYSTEM = """You are a precise academic research assistant answering questions about AI/ML research papers.
Answer ONLY from the provided context. Cite every claim with [Source N].
If context is insufficient, say: "I don't have enough information in the provided context to answer this question." """

def run_experiment_2(query_items: list, collection) -> dict:
    """EXPERIMENT 2: MOST RELEVANT FIRST VS MOST RELEVANT LAST**"""
    print("\n" + "="*60)
    print("EXPERIMENT 2 — CONTEXT ORDERING (LOST IN THE MIDDLE)")
    print("="*60)

    default_results  = []
    reversed_results = []

    for i, (qid, query, modality, gt) in enumerate(query_items):
        chunks = retrieve_reranked(query, collection, k=3)

        # CONDITION A: MOST RELEVANT FIRST**
        ctx_default,  ctx_list_default  = format_chunks(chunks, order="default")
        # CONDITION B: MOST RELEVANT LAST**
        ctx_reversed, ctx_list_reversed = format_chunks(chunks, order="reversed")

        ans_default = call_gpt(
            ORDERING_SYSTEM,
            f"CONTEXT:\n{ctx_default}\n\n---\n\nQUESTION: {query}\nAnswer only from context. Cite with [Source N]."
        )
        ans_reversed = call_gpt(
            ORDERING_SYSTEM,
            f"CONTEXT:\n{ctx_reversed}\n\n---\n\nQUESTION: {query}\nAnswer only from context. Cite with [Source N]."
        )

        default_results.append({
            "question": query, "answer": ans_default,
            "contexts": ctx_list_default, "ground_truth": gt,
            "modality": modality, "condition": "relevant_first"
        })
        reversed_results.append({
            "question": query, "answer": ans_reversed,
            "contexts": ctx_list_reversed, "ground_truth": gt,
            "modality": modality, "condition": "relevant_last"
        })

        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} done")

    scores_default  = run_ragas(default_results,  "exp2_relevant_first")
    scores_reversed = run_ragas(reversed_results, "exp2_relevant_last")

    return {
        "experiment":           "2_context_ordering",
        "hypothesis":           "Most relevant chunks first improves faithfulness (lost-in-the-middle)",
        "independent_variable": "order of retrieved chunks in context window",
        "scores":               [scores_default, scores_reversed],
        "results":              {"relevant_first": default_results,
                                 "relevant_last":  reversed_results},
    }


# ══════════════════════════════════════════════════════════
# EXPERIMENT 3 — K=3 vs K=5 vs K=10 CHUNKS
# ══════════════════════════════════════════════════════════
# HYPOTHESIS: MORE CHUNKS IMPROVES RELEVANCY BUT HURTS
# FAITHFULNESS DUE TO IRRELEVANT CONTENT DILUTING CONTEXT**
# INDEPENDENT VARIABLE: NUMBER OF RETRIEVED CHUNKS K**
# CONTROLLED: SAME QUERIES, PROMPT, MODEL, TEMPERATURE=0**

K_SYSTEM = """You are a precise academic research assistant answering questions about AI/ML research papers.
Answer ONLY from the provided context. Cite every claim with [Source N].
If context is insufficient, say: "I don't have enough information in the provided context to answer this question." """

def run_experiment_3(query_items: list, collection) -> dict:
    """EXPERIMENT 3: K=3 vs K=5 vs K=10 RETRIEVED CHUNKS**"""
    print("\n" + "="*60)
    print("EXPERIMENT 3 — K=3 vs K=5 vs K=10 CHUNKS")
    print("="*60)

    results_k3  = []
    results_k5  = []
    results_k10 = []

    for i, (qid, query, modality, gt) in enumerate(query_items):
        for k, result_list in [(3,  results_k3),
                               (5,  results_k5),
                               (10, results_k10)]:
            chunks = retrieve_reranked(query, collection, k=k)
            context_str, context_list = format_chunks(chunks, order="default")

            ans = call_gpt(
                K_SYSTEM,
                f"CONTEXT:\n{context_str}\n\n---\n\nQUESTION: {query}\nAnswer only from context. Cite with [Source N].",
                max_tokens=700
            )
            result_list.append({
                "question": query, "answer": ans,
                "contexts": context_list, "ground_truth": gt,
                "modality": modality, "condition": f"k={k}"
            })

        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} done")

    scores_k3  = run_ragas(results_k3,  "exp3_k3")
    scores_k5  = run_ragas(results_k5,  "exp3_k5")
    scores_k10 = run_ragas(results_k10, "exp3_k10")

    return {
        "experiment":           "3_chunk_count",
        "hypothesis":           "More chunks improves relevancy but reduces faithfulness due to noise",
        "independent_variable": "number of retrieved chunks (k=3, 5, 10)",
        "scores":               [scores_k3, scores_k5, scores_k10],
        "results":              {"k3":  results_k3,
                                 "k5":  results_k5,
                                 "k10": results_k10},
    }


# ══════════════════════════════════════════════════════════
# EXPERIMENT 4 — CHAIN-OF-THOUGHT vs DIRECT ANSWERING
# ══════════════════════════════════════════════════════════
# HYPOTHESIS: COT PROMPTING IMPROVES FAITHFULNESS BY FORCING
# MODEL TO REASON THROUGH EVIDENCE BEFORE ANSWERING**
# INDEPENDENT VARIABLE: REASONING STYLE IN SYSTEM PROMPT**
# CONTROLLED: SAME QUERIES, CHUNKS, MODEL, TEMPERATURE=0**

DIRECT_SYSTEM = """You are a precise academic research assistant answering questions about AI/ML research papers.
Answer ONLY from the provided context. Cite every claim with [Source N].
If context is insufficient, say: "I don't have enough information in the provided context to answer this question." """

COT_SYSTEM = """You are a precise academic research assistant answering questions about AI/ML research papers.

For EVERY question follow these steps explicitly:
STEP 1 — SCAN: Read all context chunks and identify which are relevant.
STEP 2 — EXTRACT: List the specific facts from relevant chunks that help answer the question.
STEP 3 — ANSWER: Write your final answer using ONLY extracted facts. Cite every claim with [Source N].
STEP 4 — VERIFY: Confirm every claim in your answer is directly supported by the context.

If at Step 2 you find no relevant facts respond: "I don't have enough information in the provided context to answer this question."
Do NOT use prior knowledge. Think step by step."""

def run_experiment_4(query_items: list, collection) -> dict:
    """EXPERIMENT 4: CHAIN-OF-THOUGHT vs DIRECT ANSWERING**"""
    print("\n" + "="*60)
    print("EXPERIMENT 4 — CHAIN-OF-THOUGHT vs DIRECT ANSWERING")
    print("="*60)

    direct_results = []
    cot_results    = []

    for i, (qid, query, modality, gt) in enumerate(query_items):
        # RETRIEVE ONCE — SAME CHUNKS FOR BOTH CONDITIONS**
        chunks = retrieve_reranked(query, collection, k=3)
        context_str, context_list = format_chunks(chunks, order="default")

        user_msg = (
            f"CONTEXT:\n{context_str}\n\n---\n\n"
            f"QUESTION: {query}\nAnswer only from context. Cite with [Source N]."
        )

        # CONDITION A: DIRECT — NO REASONING STEPS**
        ans_direct = call_gpt(DIRECT_SYSTEM, user_msg,
                              temperature=0.0, max_tokens=500)

        # CONDITION B: CHAIN-OF-THOUGHT — EXPLICIT REASONING**
        ans_cot = call_gpt(COT_SYSTEM, user_msg,
                           temperature=0.0, max_tokens=800)

        direct_results.append({
            "question": query, "answer": ans_direct,
            "contexts": context_list, "ground_truth": gt,
            "modality": modality, "condition": "direct"
        })
        cot_results.append({
            "question": query, "answer": ans_cot,
            "contexts": context_list, "ground_truth": gt,
            "modality": modality, "condition": "chain_of_thought"
        })

        if (i + 1) % 10 == 0:
            print(f"  ✅ {i+1}/{len(query_items)} done")

    scores_direct = run_ragas(direct_results, "exp4_direct")
    scores_cot    = run_ragas(cot_results,    "exp4_chain_of_thought")

    return {
        "experiment":           "4_chain_of_thought",
        "hypothesis":           "CoT prompting improves faithfulness by forcing evidence-based reasoning",
        "independent_variable": "reasoning style (direct vs chain-of-thought)",
        "scores":               [scores_direct, scores_cot],
        "results":              {"direct": direct_results,
                                 "cot":    cot_results},
    }


# ══════════════════════════════════════════════════════════
# PRINT SUMMARY TABLE
# ══════════════════════════════════════════════════════════
def print_summary(exp: dict):
    """PRINT FORMATTED RESULTS TABLE FOR ONE EXPERIMENT**"""
    print(f"\n{'─'*65}")
    print(f"  {exp['experiment'].upper()}")
    print(f"  Hypothesis: {exp['hypothesis']}")
    print(f"  IV: {exp['independent_variable']}")
    print(f"  {'─'*60}")
    print(f"  {'CONDITION':<30} {'FAITHFULNESS':>14} {'ANSWER_REL':>12}")
    print(f"  {'─'*60}")
    for s in exp["scores"]:
        print(f"  {s['label']:<30} {s['faithfulness']:>14.4f} "
              f"{s['answer_relevancy']:>12.4f}")
    print(f"{'─'*65}")



In [ ]:

# ══════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════

# LOAD FILES **
print("Loading benchmark files...")
with open(QUERIES_PATH, "r") as f:
    queries = json.load(f)
with open(ANSWERS_PATH, "r") as f:
    answers_gt = json.load(f)
print(f"Queries: {len(queries)} | Answers: {len(answers_gt)}\n")

# PREPARE QUERY ITEMS **
query_items = load_query_items(queries, answers_gt, limit=TEST_LIMIT)
print(f"Running 4 experiments × {len(query_items)} queries each\n")

# ── RUN ALL 4 EXPERIMENTS ─────────────────────────────────
exp1 = run_experiment_1(query_items, col_section)
exp2 = run_experiment_2(query_items, col_section)
exp3 = run_experiment_3(query_items, col_section)
exp4 = run_experiment_4(query_items, col_section)

all_experiments = [exp1, exp2, exp3, exp4]

# ── PRINT FINAL SUMMARY ───────────────────────────────────
print("\n\n" + "="*65)
print("FINAL RESULTS — ALL EXPERIMENTS")
print("Note: Faithfulness and Answer Relevancy scored by GPT-4o-mini judge")
print("="*65)
for exp in all_experiments:
    print_summary(exp)


In [ ]:
# ── SAVE ALL RESULTS ──────────────────────────────────────
output = {
    "task":        "2.3_prompt_experiments",
    "model":       GEN_MODEL,
    "collection":  "section_chunks",
    "strategy":    "reranked",
    "n_queries":   len(query_items),
    "metric_note": "GPT-4o-mini judge used for faithfulness and answer relevancy scoring",
    "experiments": [
        {
            "experiment":           e["experiment"],
            "hypothesis":           e["hypothesis"],
            "independent_variable": e["independent_variable"],
            "scores":               e["scores"],
        }
        for e in all_experiments
    ],
    "full_results": {
        e["experiment"]: e["results"]
        for e in all_experiments
    }
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f"\n✅ Saved all results to {OUTPUT_PATH}")

In [ ]:
# ── SETUP ─────────────────────────────────────────────────
GITHUB_TOKEN = "GITHUB_TOKEN_REMOVED"        # **PASTE YOUR TOKEN**
GITHUB_USER  = "RyanChenJung"           # **YOUR GITHUB USERNAME**
REPO_NAME    = "Rag-Pipeline-Over-AI-Research-Paper"
BRANCH       = "main"                   # **OR YOUR BRANCH NAME**

# YOUR FILES TO PUSH **
NOTEBOOK_PATH = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3/task2_retrieval_generation_rag.ipynb"

import os
os.chdir("/content")

# ── STEP 1: CLONE REPO ────────────────────────────────────
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git
os.chdir(f"/content/{REPO_NAME}")

# ── STEP 2: CONFIGURE GIT ─────────────────────────────────
!git config user.email "lawrence.jy.lin@gmail.com"   # **YOUR EMAIL**
!git config user.name "Lawrence Lin"         # **YOUR NAME**

# ── STEP 3: CREATE FOLDER AND COPY FILE ───────────────────
!mkdir -p part2_retrieval_generation
!cp {NOTEBOOK_PATH} part2_retrieval_generation/

# ── STEP 4: CREATE .GITIGNORE ─────────────────────────────
gitignore = """
# LARGE OUTPUT FILES — ON GOOGLE DRIVE **
task2_2_results.json
chromadb/
chunks_*.json
*.chroma
__pycache__/
.ipynb_checkpoints/
"""
with open(".gitignore", "w") as f:
    f.write(gitignore)

# ── STEP 5: CHECK WHAT WILL BE PUSHED ─────────────────────
print("Files to be pushed:")
!git status

# ── STEP 6: ADD COMMIT PUSH ───────────────────────────────
!git add part2_retrieval_generation/task2_retrieval_generation_rag.ipynb
!git add .gitignore
!git commit -m "Add Task 2.1, 2.2, 2.3 — retrieval, generation, prompt experiments"
!git push origin {BRANCH}

print("\n✅ Done! Check your GitHub repo.")

In [ ]:
# ══════════════════════════════════════════════════════════
# PUSH CODE TO GITHUB — INDUSTRY PRACTICE
# ONLY PUSHES CODE FILES, NOT DATA OR OUTPUTS
# ══════════════════════════════════════════════════════════

import os

# ── CONFIG — FILL THESE IN BEFORE RUNNING ─────────────────
GITHUB_TOKEN = "GITHUB_TOKEN_REMOVED"        # **YOUR GITHUB PERSONAL ACCESS TOKEN**
GITHUB_USER  = "RyanChenJung"           # **YOUR GITHUB USERNAME**
REPO_NAME    = "Rag-Pipeline-Over-AI-Research-Paper"
BRANCH       = "lawrence/task2-retrieval-generation"  # **YOUR BRANCH NAME**
YOUR_EMAIL   = "lawrence.jy.lin@gmail.com"         # **YOUR GITHUB EMAIL**
YOUR_NAME    = "Lawrence Lin"              # **YOUR NAME**

# ── FILES TO PUSH — ONLY NOTEBOOKS, NO DATA ───────────────
DRIVE_BASE = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3"

FILES_TO_PUSH = {
    # SOURCE FILE ON DRIVE                                    TARGET IN REPO
    f"{DRIVE_BASE}/task2_retrieval_generation_rag.ipynb" : "part2_retrieval_generation/task2_retrieval_generation_rag.ipynb"
}
# NOTE: task2_2_results.json NOT pushed — too large, stays on Drive **

# ══════════════════════════════════════════════════════════
# STEP 1 — CLONE REPO
# ══════════════════════════════════════════════════════════
os.chdir("/content")

# REMOVE OLD CLONE IF EXISTS **
!rm -rf /content/{REPO_NAME}

!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git
os.chdir(f"/content/{REPO_NAME}")

# ══════════════════════════════════════════════════════════
# STEP 2 — CONFIGURE GIT IDENTITY
# ══════════════════════════════════════════════════════════
!git config user.email "{YOUR_EMAIL}"
!git config user.name "{YOUR_NAME}"

# ══════════════════════════════════════════════════════════
# STEP 3 — SWITCH TO YOUR BRANCH
# ══════════════════════════════════════════════════════════
# CREATE BRANCH IF IT DOESN'T EXIST, OTHERWISE SWITCH TO IT **
!git checkout {BRANCH} 2>/dev/null || git checkout -b {BRANCH}

# ══════════════════════════════════════════════════════════
# STEP 4 — CREATE .GITIGNORE
# ══════════════════════════════════════════════════════════
gitignore_content = """
# LARGE DATA FILES — STORED ON GOOGLE DRIVE **
task2_2_results.json
task2_1_all_results.json
task2_1_hybrid_results.json
chromadb/
pdfs/
chunks_*.json
*.chroma

# DATASET FILES **
queries.json
answers.json
qrels.json

# PYTHON TEMP FILES **
__pycache__/
*.pyc
*.pyo

# JUPYTER TEMP FILES **
.ipynb_checkpoints/

# ENV AND SECRETS — NEVER PUSH THESE **
.env
secrets.json
config.yaml
"""
with open(".gitignore", "w") as f:
    f.write(gitignore_content)
print("✅ .gitignore created")

# ══════════════════════════════════════════════════════════
# STEP 5 — COPY FILES FROM DRIVE INTO REPO
# ══════════════════════════════════════════════════════════
for source, target in FILES_TO_PUSH.items():
    # CREATE TARGET FOLDER IF NEEDED **
    target_dir = os.path.dirname(target)
    if target_dir:
        os.makedirs(target_dir, exist_ok=True)

    # COPY FILE **
    if os.path.exists(source):
        !cp "{source}" "{target}"
        size_mb = os.path.getsize(source) / (1024 * 1024)
        print(f"✅ Copied: {os.path.basename(source)} ({size_mb:.2f} MB)")
    else:
        print(f"❌ Not found: {source}")

# ══════════════════════════════════════════════════════════
# STEP 6 — CHECK WHAT WILL BE PUSHED
# ══════════════════════════════════════════════════════════
print("\n--- Files staged for push ---")
!git status

# ══════════════════════════════════════════════════════════
# STEP 7 — ADD COMMIT PUSH
# ══════════════════════════════════════════════════════════
# STAGE FILES **
!git add .gitignore
for _, target in FILES_TO_PUSH.items():
    !git add "{target}"

# VERIFY WHAT IS STAGED **
print("\n--- Staged files ---")
!git diff --cached --name-only

# COMMIT **
!git commit -m "Add Task 2.1 2.2 2.3 — retrieval, generation, prompt experiments"

# PUSH **
!git push origin {BRANCH}

print(f"\n✅ Done! Check: https://github.com/{GITHUB_USER}/{REPO_NAME}/tree/{BRANCH}")

In [ ]:
# SEARCH FOR YOUR NOTEBOOK ON DRIVE **
import os

search_dir = "/content/drive/MyDrive/[3Q]Capstone1/Assignment3"

print("Files in your Assignment3 folder:")
for f in os.listdir(search_dir):
    print(f"  {f}")